# Parte 2a: Modelo A (CNN custom)

> **Notebook auxiliar.** Se usa para trabajar en paralelo. Su contenido se integra en `2-train.ipynb`, que es la que se entrega.
> La lógica va en `src/lib/`; acá solo se llaman funciones. Al terminar, guarda los embeddings con `save_embeddings` (contrato en `src/lib/evaluation/embedding_store.py`).

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.evaluation import EmbeddingSet, load_embeddings, results_path, save_embeddings

CONFIG = json.loads((ROOT / "splits" / "config.json").read_text(encoding="utf-8"))
ALIGNED_ROOT = ROOT / CONFIG["aligned_root"]
SEED = CONFIG["seed"]
splits = pd.read_csv(ROOT / "splits" / "splits.csv")
splits["split"].value_counts()

In [ ]:
def load_faces(rows: pd.DataFrame) -> list[np.ndarray]:
    """Caras alineadas en BGR, tal como las guardó la Parte 1."""
    return [cv2.imread(str(ALIGNED_ROOT / p)) for p in rows["aligned_path"]]

## 1. Datos
- `train`: entrenamiento. `val_closed`: curvas, early stopping y mejor checkpoint.
- Augmentation de la Parte 1: `lib.dataset.augmentation.build_train_augmentation`.
- Mismo preprocesamiento que la app: `lib.vision.InputNormalization`.

In [ ]:
# TODO: Dataset / DataLoader de train y val_closed

## 2. Arquitectura
Capas, cantidad de parámetros, dimensión del embedding y `summary`. Justificar profundidad, BatchNorm, Dropout y global pooling.
La clase va en `src/lib/models/custom_cnn.py` (tiene que ser importable desde el backend).

In [ ]:
# TODO: from lib.models.custom_cnn import ...  y summary del modelo

## 3. Hiperparámetros
Optimizador, learning rate y scheduler, batch size, épocas, regularización y pérdida (softmax o ArcFace / triplet).

## 4. Entrenamiento
Guardar el mejor checkpoint según validación y el historial en `results/custom_cnn/history.json`.

In [ ]:
# TODO: loop de entrenamiento con early stopping

## 5. Curvas de entrenamiento
Loss y accuracy de train vs. validation. Discusión de overfitting / underfitting.

In [ ]:
# TODO: graficar desde results/custom_cnn/history.json

## 6. Experimentos

| # | Cambio | Val loss | Val acc | Conclusión |
|---|---|---|---|---|
| 1 | | | | |

## 7. Exportación
`torch.onnx.export` a `models/custom_cnn.onnx` (no se versiona; subir a Drive/HF y poner el link en el README).

In [ ]:
# TODO: exportar a ONNX

## 8. Embeddings para la evaluación
Se calculan sobre todos los splits salvo `train` y se guardan con el contrato común.

In [ ]:
eval_rows = splits[splits["split"] != "train"].reset_index(drop=True)
embeddings = ...  # TODO: (N, D) float32 normalizados L2 con el modelo entrenado (ej. OnnxFaceEmbedder sobre el .onnx exportado)

save_embeddings(
    EmbeddingSet(
        model="custom_cnn",
        paths=eval_rows["aligned_path"].to_numpy(),
        labels=eval_rows["identity"].to_numpy(),
        splits=eval_rows["split"].to_numpy(),
        embeddings=embeddings,
    ),
    results_path(ROOT, "custom_cnn"),
)